In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import batman
import emcee
import corner

# load K2 data
data = np.loadtxt(
    "../data/apjladf06ft3_mrt.txt",
    skiprows=15
)

t = data[:, 0] + 2454833  # offset to full BJD
flux = data[:, 2]

# published mid-transit time from Venner et al. (2026)
t0_published = 2458055.0969

# only use the region around the transit
mask = np.abs(t - t0_published) < 0.8

t_transit = t[mask]
flux_transit = flux[mask]

In [ ]:
# gravitational constant in SI units
G = 6.67430e-11

# standard deviation of the out-of-transit K2 light curve
sigma_flux = 32e-6

# stellar-density prior from Venner et al. (2026):
# log10(rho_star / (g cm^-3)) = 0.462 +/- 0.041
logrho_mean = 0.462
logrho_sigma = 0.041

# limb-darkening priors in the Kepler band
u1_mean = 0.60
u1_sigma = 0.07

u2_mean = 0.12
u2_sigma = 0.07

# maximum eccentricity for this first broad-prior experiment
e_max = 0.8

In [ ]:
def a_over_rstar(period, rho_star):
    """
    Calculate a/R_star using Kepler's third law.

    period   : orbital period in days
    rho_star : stellar density in g/cm^3
    """

    # days -> seconds
    period_seconds = period * 24 * 60 * 60

    # g/cm^3 -> kg/m^3
    rho_si = rho_star * 1000

    return (
        G * rho_si * period_seconds**2
        / (3 * np.pi)
    )**(1/3)

print(a_over_rstar(355, 2.90))

In [ ]:
def eccentricity_from_hk(h, k):
    """
    Convert h = sqrt(e) cos(omega) and
    k = sqrt(e) sin(omega) back to e and omega.
    """

    e = h**2 + k**2

    # arctan2 gives omega between -pi and pi
    omega = np.degrees(np.arctan2(k, h))

    # batman prefers angles between 0 and 360 degrees
    omega = omega % 360

    return e, omega


print(eccentricity_from_hk(0.0, np.sqrt(0.1)))

In [ ]:
# starting parameters only needed to initialise batman
batman_params = batman.TransitParams()

batman_params.t0 = t0_published
batman_params.per = 355.0
batman_params.rp = 0.01368
batman_params.a = 270.0
batman_params.inc = 90.0

batman_params.ecc = 0.0
batman_params.w = 90.0

batman_params.u = [0.60, 0.12]
batman_params.limb_dark = "quadratic"

# same 29.4-min K2 exposure and 10x supersampling as before
batman_model = batman.TransitModel(
    batman_params,
    t_transit,
    supersample_factor=10,
    exp_time=29.4 / (24 * 60)
)

In [ ]:
def transit_model(theta):
    """
    Transit model with eccentricity.

    theta =
    (period, b, rp, t0, rho_star, u1, u2, h, k)
    """

    period, b, rp, t0, rho_star, u1, u2, h, k = theta

    # convert h and k back to e and omega
    e, omega = eccentricity_from_hk(h, k)

    # calculate a/Rstar from period and stellar density
    a_Rstar = a_over_rstar(period, rho_star)

    omega_rad = np.radians(omega)

    # For an eccentric orbit:
    #
    # b = (a/Rstar) cos(i) (1-e^2)/(1+e sin(omega))
    #
    # therefore:
    cos_i = (
        b / a_Rstar
        * (1 + e * np.sin(omega_rad))
        / (1 - e**2)
    )

    # reject geometrically impossible solutions
    if cos_i < 0 or cos_i > 1:
        return None

    inclination = np.degrees(np.arccos(cos_i))

    # update batman parameters
    batman_params.t0 = t0
    batman_params.per = period
    batman_params.rp = rp
    batman_params.a = a_Rstar
    batman_params.inc = inclination

    batman_params.ecc = e
    batman_params.w = omega

    batman_params.u = [u1, u2]

    return batman_model.light_curve(batman_params)

In [ ]:
#prior

def log_prior(theta):
    """
    Priors for the first eccentric model.

    For now we use a broad uniform eccentricity prior:
    0 <= e < 0.8.

    The other priors are kept the same as in the circular model
    so that we can make a fair comparison.
    """

    period, b, rp, t0, rho_star, u1, u2, h, k = theta

    # convert h,k -> eccentricity and omega
    e, omega = eccentricity_from_hk(h, k)


    # --------------------------------------------------
    # Basic parameter limits
    # --------------------------------------------------

    if not 64 < period < 2000:
        return -np.inf

    if not 0 <= b < 1 + rp:
        return -np.inf

    if not 0.005 < rp < 0.03:
        return -np.inf

    if not t0_published - 0.1 < t0 < t0_published + 0.1:
        return -np.inf

    if rho_star <= 0:
        return -np.inf


    # --------------------------------------------------
    # Eccentricity prior
    # --------------------------------------------------

    # Since e = h^2 + k^2, this restricts h and k to a disk.
    # Uniform sampling inside this disk corresponds to
    # uniform e and uniform omega.
    if not 0 <= e < e_max:
        return -np.inf


    # --------------------------------------------------
    # Physical limb-darkening conditions
    # --------------------------------------------------

    if not (
        u1 > 0
        and u1 + u2 < 1
        and u1 + 2*u2 > 0
    ):
        return -np.inf


    # --------------------------------------------------
    # Orbital geometry
    # --------------------------------------------------

    a_Rstar = a_over_rstar(period, rho_star)

    # Reject solutions where the planet's periastron
    # would actually be inside the star.
    if a_Rstar * (1 - e) <= 1 + rp:
        return -np.inf

    omega_rad = np.radians(omega)

    cos_i = (
        b / a_Rstar
        * (1 + e * np.sin(omega_rad))
        / (1 - e**2)
    )

    if cos_i < 0 or cos_i > 1:
        return -np.inf


    lp = 0.0


    # --------------------------------------------------
    # Same period prior as in notebook 03
    # --------------------------------------------------

    # log-uniform period prior
    lp += -np.log(period)

    # approximation of the Kipping single-transit prior
    lp += -(5/3) * np.log(period)


    # --------------------------------------------------
    # Radius-ratio prior
    # --------------------------------------------------

    lp += -np.log(rp)


    # --------------------------------------------------
    # Stellar-density prior
    # --------------------------------------------------

    lp += -np.log(rho_star)

    log_rho = np.log10(rho_star)

    lp += -0.5 * (
        (log_rho - logrho_mean) / logrho_sigma
    )**2


    # --------------------------------------------------
    # Limb-darkening priors
    # --------------------------------------------------

    lp += -0.5 * ((u1 - u1_mean) / u1_sigma)**2
    lp += -0.5 * ((u2 - u2_mean) / u2_sigma)**2


    # No extra eccentricity term is needed here.
    # A uniform distribution in the h-k disk gives us
    # a uniform prior in e and omega.

    return lp

In [ ]:
#likelihood

def log_likelihood(theta):
    """
    Gaussian likelihood comparing the observed K2 flux
    with the eccentric transit model.
    """

    model_flux = transit_model(theta)

    if model_flux is None:
        return -np.inf

    if not np.all(np.isfinite(model_flux)):
        return -np.inf

    residuals = flux_transit - model_flux

    return -0.5 * np.sum(
        residuals**2 / sigma_flux**2
        + np.log(2 * np.pi * sigma_flux**2)
    )

In [ ]:
#posterior

def log_probability(theta):
    """
    log posterior = log prior + log likelihood
    """

    lp = log_prior(theta)

    if not np.isfinite(lp):
        return -np.inf

    return lp + log_likelihood(theta)

In [ ]:
#small test run to check that the code works

e_test = 0.05
omega_test = 90.0

h_test = np.sqrt(e_test) * np.cos(np.radians(omega_test))
k_test = np.sqrt(e_test) * np.sin(np.radians(omega_test))

theta_test = (
    355.0,          # period [days]
    0.3,            # impact parameter
    0.01368,        # Rp/Rstar
    t0_published,   # mid-transit time
    2.90,           # stellar density
    0.60,           # u1
    0.12,           # u2
    h_test,
    k_test
)

print("e, omega:", eccentricity_from_hk(h_test, k_test))
print("log prior:", log_prior(theta_test))
print("log likelihood:", log_likelihood(theta_test))
print("log posterior:", log_probability(theta_test))

In [ ]:
model_test = transit_model(theta_test)

plt.figure(figsize=(9, 5))

plt.plot(
    (t_transit - t0_published) * 24,
    flux_transit,
    ".",
    label="K2 data"
)

plt.plot(
    (t_transit - t0_published) * 24,
    model_test,
    label="Eccentric test model"
)

plt.xlabel("Time from mid-transit [hours]")
plt.ylabel("Detrended flux")
plt.legend()
plt.show()

In [ ]:
#mcmc

ndim = 9
nwalkers = 60

labels = [
    "Period [d]",
    "b",
    "Rp/Rstar",
    "t0 [BJD]",
    "rho_star",
    "u1",
    "u2",
    "h",
    "k"
]

initial = np.array([
    355.0,
    0.3,
    0.01368,
    t0_published,
    2.90,
    0.60,
    0.12,
    h_test,
    k_test
])

spread = np.array([
    30.0,      # period
    0.15,      # b
    0.0005,    # Rp/Rstar
    0.005,     # t0
    0.10,      # rho_star
    0.03,      # u1
    0.03,      # u2
    0.08,      # h
    0.08       # k
])

rng = np.random.default_rng(42)

In [ ]:
pos = []
#generating only valid starting walkers
while len(pos) < nwalkers:

    candidate = initial + spread * rng.normal(size=ndim)

    if np.isfinite(log_probability(candidate)):
        pos.append(candidate)

pos = np.array(pos)

print(pos.shape)

In [ ]:
#mcmc run

sampler = emcee.EnsembleSampler(
    nwalkers,
    ndim,
    log_probability
)

sampler.run_mcmc(
    pos,
    10000,
    progress=True
)

In [ ]:
print(sampler.get_chain().shape)

In [ ]:
#diagnostics

print(
    "Mean acceptance fraction:",
    np.mean(sampler.acceptance_fraction)
)

tau = sampler.get_autocorr_time(tol=0)

print()

for label, value in zip(labels, tau):
    print(f"{label}: tau = {value:.1f}")

print()
print("Chain length / tau:")

for label, value in zip(labels, tau):
    print(
        f"{label}: "
        f"{sampler.iteration / value:.1f}"
    )

In [ ]:
chain = sampler.get_chain()

fig, axes = plt.subplots(
    ndim,
    figsize=(10, 16),
    sharex=True
)

for i in range(ndim):

    axes[i].plot(
        chain[:, :, i],
        alpha=0.20
    )

    axes[i].set_ylabel(labels[i])

axes[-1].set_xlabel("MCMC step")

plt.tight_layout()
plt.show()

In [ ]:
#more steps

sampler.run_mcmc(
    None,
    10000,
    progress=True
)

In [ ]:
tau = sampler.get_autocorr_time(tol=0)

for label, value in zip(labels, tau):
    print(
        f"{label}: tau = {value:.1f}, "
        f"N/tau = {sampler.iteration/value:.1f}"
    )

In [ ]:
#stability check

chain = sampler.get_chain()

N = sampler.iteration

# ignore the first quarter and compare two later pieces
start = N // 4
middle = (start + N) // 2

samples_earlier = chain[start:middle].reshape(-1, ndim)
samples_later = chain[middle:N].reshape(-1, ndim)

for i, label in enumerate(labels):

    q_earlier = np.percentile(
        samples_earlier[:, i],
        [16, 50, 84]
    )

    q_later = np.percentile(
        samples_later[:, i],
        [16, 50, 84]
    )

    print(label)
    print("  earlier:", q_earlier)
    print("  later:  ", q_later)

In [ ]:
#posterior

burnin = sampler.iteration // 2
thin = 100

flat_samples = sampler.get_chain(
    discard=burnin,
    thin=thin,
    flat=True
)

print("Total chain length:", sampler.iteration)
print("Burn-in:", burnin)
print("Posterior samples:", len(flat_samples))

In [ ]:
np.save(
    "../data/eccentric_uniform_posterior.npy",
    flat_samples
)

In [ ]:
#corner plot of sampled parameters

fig = corner.corner(
    flat_samples,
    labels=labels,
    show_titles=True,
    quantiles=[0.16, 0.50, 0.84]
)

plt.show()

In [ ]:
h_samples = flat_samples[:, 7]
k_samples = flat_samples[:, 8]

e_samples = h_samples**2 + k_samples**2

omega_samples = (
    np.degrees(
        np.arctan2(k_samples, h_samples)
    )
    % 360
)

print("Median eccentricity:", np.median(e_samples))

In [ ]:
plt.figure(figsize=(7, 4))

plt.hist(
    e_samples,
    bins=40,
    density=True,
    histtype="step"
)

plt.xlabel("Eccentricity e")
plt.ylabel("Posterior density")
plt.title("Eccentricity posterior")

plt.show()

In [ ]:
period_samples = flat_samples[:, 0]

plt.figure(figsize=(7, 5))

plt.scatter(
    e_samples,
    period_samples,
    s=5,
    alpha=0.15
)

plt.xlabel("Eccentricity e")
plt.ylabel("Period [days]")
plt.title("Period-eccentricity degeneracy")

plt.show()

In [ ]:
science_samples = np.column_stack([
    flat_samples[:, 0],   # period
    flat_samples[:, 1],   # b
    e_samples,
    omega_samples
])

science_labels = [
    "Period [d]",
    "b",
    "e",
    "omega [deg]"
]

fig = corner.corner(
    science_samples,
    labels=science_labels,
    show_titles=True,
    quantiles=[0.16, 0.50, 0.84]
)

plt.show()

In [ ]:
#posterior predictive check

rng = np.random.default_rng(1)

n_models = min(100, len(flat_samples))

indices = rng.choice(
    len(flat_samples),
    size=n_models,
    replace=False
)

plt.figure(figsize=(9, 5))

# draw posterior models first
for index in indices:

    model_flux = transit_model(
        flat_samples[index]
    )

    plt.plot(
        (t_transit - t0_published) * 24,
        model_flux,
        alpha=0.05
    )

# observations on top
plt.plot(
    (t_transit - t0_published) * 24,
    flux_transit,
    ".",
    label="K2 data"
)

plt.xlabel("Time from mid-transit [hours]")
plt.ylabel("Detrended flux")
plt.title("Eccentric-model posterior predictive check")
plt.legend()

plt.show()

In [ ]:
#numerical results

for i, label in enumerate(labels[:7]):

    q16, q50, q84 = np.percentile(
        flat_samples[:, i],
        [16, 50, 84]
    )

    print(
        f"{label}: "
        f"{q50:.5f} "
        f"-{q50-q16:.5f} "
        f"+{q84-q50:.5f}"
    )

In [ ]:
e16, e50, e84 = np.percentile(
    e_samples,
    [16, 50, 84]
)

print(
    f"e: "
    f"{e50:.3f} "
    f"-{e50-e16:.3f} "
    f"+{e84-e50:.3f}"
)

In [ ]:
np.save("../data/circular_posterior.npy", flat_samples)

In [ ]:
circular_samples = np.load(
    "../data/circular_posterior.npy"
)

period_circular = circular_samples[:, 0]
period_eccentric = flat_samples[:, 0]

In [ ]:
plt.figure(figsize=(8, 5))

plt.hist(
    period_circular,
    bins=50,
    density=True,
    histtype="step",
    label="Circular: e = 0"
)

plt.hist(
    period_eccentric,
    bins=50,
    density=True,
    histtype="step",
    label="Broad eccentricity prior"
)

plt.xlabel("Orbital period [days]")
plt.ylabel("Posterior density")
plt.title("Effect of allowing eccentricity")

plt.xlim(50, 1500)

plt.legend()
plt.show()

In [ ]:
def summary(samples):

    q16, q50, q84 = np.percentile(
        samples,
        [16, 50, 84]
    )

    return q50, q50-q16, q84-q50


circ = summary(period_circular)
ecc = summary(period_eccentric)

print(
    f"Circular: "
    f"{circ[0]:.1f} "
    f"-{circ[1]:.1f} "
    f"+{circ[2]:.1f} days"
)

print(
    f"Eccentric: "
    f"{ecc[0]:.1f} "
    f"-{ecc[1]:.1f} "
    f"+{ecc[2]:.1f} days"
)

In [ ]:
#add more steps

sampler.run_mcmc(
    None,
    10000,
    progress=True
)

In [ ]:
circular_samples = np.load("../data/circular_posterior.npy")

print(circular_samples.shape)
print(np.percentile(circular_samples[:, 0], [16, 50, 84]))

In [ ]:
np.save(
    "../data/posterior_eccentric_uniform.npy",
    flat_samples
)

In [ ]:
circular_samples = np.load(
    "../data/posterior_circular.npy"
)

eccentric_samples = np.load(
    "../data/posterior_eccentric_uniform.npy"
)

period_circular = circular_samples[:, 0]
period_eccentric = eccentric_samples[:, 0]

In [ ]:
print(
    "Circular:",
    np.percentile(period_circular, [16, 50, 84])
)

print(
    "Eccentric:",
    np.percentile(period_eccentric, [16, 50, 84])
)

In [ ]:
bins = np.linspace(50, 1000, 80)

plt.figure(figsize=(8, 5))

plt.hist(
    period_circular,
    bins=bins,
    density=True,
    histtype="step",
    linewidth=2,
    label="Circular: e = 0"
)

plt.hist(
    period_eccentric,
    bins=bins,
    density=True,
    histtype="step",
    linewidth=2,
    label="Broad eccentricity prior"
)

plt.xlabel("Orbital period [days]")
plt.ylabel("Posterior density")
plt.title("Effect of allowing eccentricity")
plt.xlim(50, 1000)
plt.legend()
plt.show()

In [ ]:
np.save(
    "../data/posterior_eccentric_uniform.npy",
    flat_samples
)